# Finding and understanding data across domains: a dwarf tomato example

*DigiLab demonstrator: metadata catalogue and vocabulary hub*

**Imagine a federated catalogue.** Researchers today usually start in their own domain repository. Imagine instead one catalogue that harvests descriptions from many repositories (agriculture, environment, biodiversity) and a vocabulary hub that explains what the terms in those descriptions mean. The data itself stays where it is.

A researcher (or an AI agent working for them) asks:

> *Find datasets with plant growth and morphology measurements of tomato under controlled conditions, and any related climate or specimen data. Show which concepts are related, which measurements may be comparable, and what extra context I need to interpret them.*

This notebook walks through that question in seven steps:

1. Search the catalogue
2. What's in this dataset?
3. What do the terms mean?
4. What was measured, and under which conditions?
5. How are terms linked, and who says so?
6. Find related data in other domains
7. What can be compared, and what context is missing?

It ends with a picture of how everything connects and a simple architecture view.

*This is a showcase of the concepts, not a finished tool. The same user journey can be rerun with other keywords, datasets or domains to test other cases.*

<details><summary><b>Behind the scenes: what stands in for what</b></summary>

| Architecture component | Stand-in for this demo |
|---|---|
| Federated metadata catalogue (DCAT/CDIF) | **Simulated** with the DataCite search API, restricted to 4TU.ResearchData |
| Dataset description (RO-Crate) | The RO-Crate that 4TU.ResearchData publishes for each dataset |
| Data repository | 4TU.ResearchData file API |
| Vocabulary hub (terms) | EBI Ontology Lookup Service (Plant Trait Ontology, PATO, ENVO, ChEBI, NCBI Taxonomy) |
| Vocabulary hub (mappings) | An SSSOM mapping file built in this notebook |
| Other domains | GBIF (living specimens), DataCite (climate data) |

If an API cannot be reached, the notebook falls back to a short cached description and says so.
</details>

**How to run:** choose *Run All* (internet access needed). Each step calls live services, so it takes a few seconds per cell; the whole notebook usually runs in under a minute. The code is hidden; click the three dots (or the bar left of a cell) to show it.

### Setup

In [ ]:
# If needed: %pip install requests pandas openpyxl pyyaml matplotlib
import json, re, io, textwrap
from pathlib import Path
import requests
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from matplotlib.lines import Line2D
from IPython.display import display, HTML, Markdown

pd.set_option("display.max_colwidth", 80)

# ---------------- Demo configuration ----------------
SEARCH_QUERIES = ['"dwarf tomato"', 'tomato AND "vertical farm"', 'tomato AND greenhouse AND (yield OR crop OR plant)']
REPOSITORY_PREFIX = "10.4121"                                   # 4TU.ResearchData
PRIMARY_DOI    = "10.4121/1ab846cb-c590-4220-9da7-f17e2217f96c"  # vertical farm trials (WUR)
COMPARISON_DOI = "10.4121/e1ee9de9-6ce9-4502-a37c-34b5b1372bed"  # greenhouse pre-trial (WUR)
SELECTED = [PRIMARY_DOI, COMPARISON_DOI]
LABEL = {PRIMARY_DOI: "Vertical farm trials", COMPARISON_DOI: "Greenhouse pre-trial"}
SHORT = {PRIMARY_DOI: "vf", COMPARISON_DOI: "gh"}
MAX_BYTES = 15_000_000        # do not download big files just to read headers
OUT = Path("demo_output"); OUT.mkdir(exist_ok=True)
TIMEOUT = 25
HTTP = requests.Session()
HTTP.headers["User-Agent"] = "DigiLab-catalogue-demo/0.2"

# ---------------- Display helpers ----------------
def link(text, url):
    return f'<a href="{url}" target="_blank">{text}</a>' if url else str(text)

def show(df, header=True):
    display(HTML(df.to_html(escape=False, index=False, na_rep="", header=header)))

def details(summary, html):
    display(HTML(f"<details><summary><i>{summary}</i></summary>{html}</details>"))

# ---------------- Network helpers ----------------
def get_json(url, params=None):
    try:
        r = HTTP.get(url, params=params, timeout=TIMEOUT)
        r.raise_for_status()
        return r.json()
    except Exception as e:
        print(f"  ! could not reach {url.split('/')[2]} ({type(e).__name__})")
        return None

def is_online():
    try:
        return HTTP.get("https://api.datacite.org/heartbeat", timeout=10).ok
    except Exception:
        return False

ONLINE = is_online()

# ---------------- Catalogue (DataCite as stand-in) ----------------
DATACITE = "https://api.datacite.org/dois"

def datacite_search(query, prefix=None, size=25):
    params = {"query": query, "page[size]": size, "resource-type-id": "dataset", "affiliation": "true"}
    if prefix:
        params["prefix"] = prefix
    return ((get_json(DATACITE, params) if ONLINE else None) or {}).get("data", [])

def datacite_get(doi):
    return ((get_json(f"{DATACITE}/{doi}", {"affiliation": "true"}) if ONLINE else None) or {}).get("data")

def summarise(rec):
    a = rec["attributes"]
    pub = a.get("publisher"); pub = pub.get("name") if isinstance(pub, dict) else pub
    orgs = []
    for c in a.get("creators", []):
        for aff in c.get("affiliation", []) or []:
            name = aff.get("name") if isinstance(aff, dict) else aff
            if name and name not in orgs:
                orgs.append(name)
    desc = [d.get("description", "") for d in a.get("descriptions", []) if d.get("description")]
    return {"doi": a["doi"].lower(), "title": (a.get("titles") or [{}])[0].get("title"),
            "organisation": "; ".join(orgs[:2]) or pub, "publisher": pub, "year": a.get("publicationYear"),
            "url": a.get("url") or f"https://doi.org/{a['doi']}", "description": desc[0] if desc else "",
            "licence": "; ".join((r.get("rights") or r.get("rightsIdentifier") or "") for r in a.get("rightsList", []) or []),
            "licence_html": "; ".join(link(r.get("rights") or r.get("rightsIdentifier") or "licence", r.get("rightsUri"))
                                      for r in a.get("rightsList", []) or []),
            "creators": [c.get("name") for c in a.get("creators", [])]}

CACHED = {
    PRIMARY_DOI: {"doi": PRIMARY_DOI,
        "title": "Dwarf tomato trials in a research vertical farm facility: climate, yield and crop related data",
        "organisation": "Wageningen University & Research", "publisher": "4TU.ResearchData", "year": 2026,
        "url": f"https://doi.org/{PRIMARY_DOI}", "licence": "",
        "description": ("Two trials (2021, 2022) in two climate-controlled chambers of the experimental vertical farm "
                        "in Bleiswijk (NL); 14 datasets combining cultivation system, daylight integral, temperature "
                        "and year; cultivars F12414 (red, plum) and F13469B (orange, cherry)."),
        "creators": ["Righini, Isabella", "Hemming, Silke", "Stanghellini, Cecilia", "Graamans, Luuk", "Marcelis, Leo"]},
    COMPARISON_DOI: {"doi": COMPARISON_DOI,
        "title": "4th Autonomous Greenhouse Challenge: Pre-trial Dwarf Tomato Measurements and Images",
        "organisation": "Wageningen University & Research", "publisher": "4TU.ResearchData", "year": 2024,
        "url": f"https://doi.org/{COMPARISON_DOI}", "licence": "",
        "description": ("Two-month dwarf tomato cultivation in a 96 m2 research greenhouse compartment in Bleiswijk "
                        "(NL); raw and processed data (xlsx, png)."),
        "creators": ["Wageningen University & Research"]},
}

# ---------------- Repository (4TU) ----------------
CACHED_FILES = {
    PRIMARY_DOI: [{"name": "IndoorClimate.csv", "note": "realised climate control settings"},
                  {"name": "CropParameters.csv", "note": "crop parameters from destructive measurements"},
                  {"name": "Production.csv", "note": "fruit yield and fruit dry matter content"}],
    COMPARISON_DOI: [],
}

CRATE_VERSION = 1   # 4TU dataset version to read

def as_list(x):
    return x if isinstance(x, list) else ([] if x is None else [x])

def fourtu_crate(doi):
    """The dataset's own RO-Crate, published by 4TU (None if unreachable)."""
    uuid = doi.split("/")[1]
    return get_json(f"https://data.4tu.nl/v3/datasets/{uuid}/versions/{CRATE_VERSION}/ro-crate-metadata.json") if ONLINE else None

class Crate:
    """Minimal reader for an RO-Crate 1.1 JSON-LD graph."""
    def __init__(self, data):
        self.g = {e.get("@id"): e for e in (data or {}).get("@graph", [])}
        desc = next((e for i, e in self.g.items() if str(i).endswith("ro-crate-metadata.json")), {})
        self.root = self.g.get(self.ref(desc.get("about")) or "./", {})
    @staticmethod
    def ref(x):
        return x.get("@id") if isinstance(x, dict) else x
    def entity(self, x):
        return self.g.get(self.ref(x), x if isinstance(x, dict) else {"@id": x})
    def name(self, x):
        names = []
        for item in as_list(x):
            e = self.entity(item)
            names.append(str(e.get("name") or e.get("@value") or e.get("@id") or item))
        return "; ".join(n for n in names if n)
    def people(self):
        out = []
        for p in as_list(self.root.get("author") or self.root.get("creator")):
            e = self.entity(p)
            aff = self.name(e.get("affiliation")) if e.get("affiliation") else ""
            out.append(f"{e.get('name', self.ref(p))}" + (f" ({aff})" if aff else ""))
        return out
    def licence(self):
        lic = self.root.get("license")
        if not lic:
            return None, None
        e = self.entity(as_list(lic)[0])
        url = e.get("@id") if str(e.get("@id", "")).startswith("http") else e.get("url")
        return e.get("name") or url or str(lic), url
    def files(self, node=None, depth=0):
        out = []
        for p in as_list((node or self.root).get("hasPart")):
            e = self.entity(p)
            types = as_list(e.get("@type"))
            if "File" in types:
                fid = str(e.get("@id", ""))
                size = e.get("contentSize")
                try: size = int(size)
                except (TypeError, ValueError): size = None
                out.append({"name": e.get("name") or fid.rstrip("/").split("/")[-1], "size": size,
                            "download_url": e.get("contentUrl") or (fid if fid.startswith("http") else None),
                            "format": e.get("encodingFormat"), "note": e.get("description", "")})
            elif "Dataset" in types and depth < 2:
                out += self.files(e, depth + 1)
        return out

CRATE_SHOWN = {"name", "description", "author", "creator", "license", "hasPart", "@id", "@type",
               "datePublished", "publisher", "keywords", "identifier"}

def dataset_card(crate, summary):
    """Researcher-facing card: the main facts, then every other property the crate carries."""
    r = crate.root
    lic_name, lic_url = crate.licence()
    rows = [("Title", r.get("name") or summary["title"]),
            ("Published", f"{r.get('datePublished', summary['year'])} by {crate.name(r.get('publisher')) or summary['publisher']}"),
            ("Creators", "; ".join(crate.people()) or "; ".join(summary.get("creators", []))),
            ("Licence", link(lic_name, lic_url) if lic_name else (summary.get("licence_html") or "not stated: ask before reuse")),
            ("Keywords", ", ".join(map(str, as_list(r.get("keywords")))) if r.get("keywords") else "")]
    for k, v in r.items():
        if k not in CRATE_SHOWN:
            label = re.sub(r"(?<!^)(?=[A-Z])", " ", k).capitalize()
            rows.append((label, crate.name(v) if isinstance(v, (dict, list)) else str(v)))
    return pd.DataFrame([{"field": k, "value": v} for k, v in rows if v])

def fourtu_files_v2(doi):
    files = get_json(f"https://data.4tu.nl/v2/articles/{doi.split('/')[1]}/files") if ONLINE else None
    if isinstance(files, dict):
        files = files.get("files", [])
    return files or []

def remote_size(url):
    try:
        return int(HTTP.head(url, timeout=TIMEOUT, allow_redirects=True).headers.get("Content-Length", 0)) or None
    except Exception:
        return None

def read_head(f):
    """For a small CSV or Excel file: {label: first rows as a DataFrame}. Never downloads big files."""
    url, name = f.get("download_url"), f.get("name", "")
    if not url:
        return {}
    size = f.get("size") or remote_size(url) or 0
    if size > MAX_BYTES:
        return {}
    try:
        if name.lower().endswith((".csv", ".txt", ".tsv")):
            raw = HTTP.get(url, timeout=TIMEOUT, headers={"Range": "bytes=0-200000"}).content
            text = raw.decode("utf-8-sig", errors="replace")
            if len(raw) >= 200000:
                text = text.rsplit("\n", 1)[0]
            return {name: pd.read_csv(io.StringIO(text), sep=None, engine="python", nrows=5)}
        if name.lower().endswith((".xlsx", ".xls")):
            sheets = pd.read_excel(io.BytesIO(HTTP.get(url, timeout=TIMEOUT).content), sheet_name=None, nrows=5)
            return {f"{name} [{sh}]": d for sh, d in sheets.items()}
    except Exception as e:
        print(f"  ! could not preview {name} ({type(e).__name__})")
    return {}

def describe_file(f, heads):
    """'What's in it' text: columns if previewed, otherwise the best description available."""
    name = f.get("name", "")
    cols = [str(c) for k, d in heads.items() if k == name or k.startswith(name + " [") for c in d.columns]
    if cols:
        return f"{len(cols)} columns: " + ", ".join(cols[:10]) + (" …" if len(cols) > 10 else "")
    if f.get("note"):
        return f["note"]
    if (f.get("size") or 0) > MAX_BYTES:
        return "large file: not previewed"
    ext = name.rsplit(".", 1)[-1].lower() if "." in name else ""
    kinds = {"png": "image", "jpg": "image", "jpeg": "image", "tif": "image", "zip": "archive",
             "pdf": "document", "txt": "text", "md": "documentation"}
    return kinds.get(ext, f.get("format") or "not previewed")

def licence_label(url):
    u = str(url or "").lower()
    for key, label in [("publicdomain/zero", "CC0"), ("by-nc-sa", "CC BY-NC-SA"), ("by-nc", "CC BY-NC"),
                       ("by-sa", "CC BY-SA"), ("licenses/by", "CC BY")]:
        if key in u:
            return label
    return url or "not stated"

def human_size(n):
    if not n:
        return ""
    for unit in ["B", "kB", "MB", "GB"]:
        if n < 1000:
            return f"{n:.0f} {unit}"
        n /= 1000
    return f"{n:.1f} TB"

# ---------------- Vocabulary hub (terms) ----------------
OLS = "https://www.ebi.ac.uk/ols4/api/search"
CONCEPTS_WANTED = {   # key: (search label, ontologies, cached CURIE used only offline)
    "plant_height":       ("plant height", "to", "TO:0000207"),
    "leaf_area_index":    ("leaf area index", "to,envo", None),
    "leaf_area":          ("leaf area", "to,pato", None),
    "fruit_yield":        ("fruit yield", "to", None),
    "dry_matter_content": ("dry matter content", "to,pato", None),
    "plant_dry_weight":   ("plant dry weight", "to", None),
    "temperature":        ("temperature", "pato", "PATO:0000146"),
    "relative_humidity":  ("relative humidity", "envo,pato", None),
    "carbon_dioxide":     ("carbon dioxide", "chebi", "CHEBI:16526"),
    "par":                ("photosynthetically active radiation", "envo,to,pato", None),
    "ghg_emission":       ("greenhouse gas emission", "envo", None),
    "tomato":             ("Solanum lycopersicum", "ncbitaxon", "NCBITaxon:4081"),
}

def purl(curie):
    if not curie or curie.startswith("demo:"):
        return None
    p, i = curie.split(":", 1)
    return f"http://purl.obolibrary.org/obo/{p}_{i}"

def ols_lookup(label, ontologies):
    data = get_json(OLS, {"q": label, "ontology": ontologies, "rows": 10}) if ONLINE else None
    docs = (data or {}).get("response", {}).get("docs", [])
    if not docs:
        return None
    exact = [d for d in docs if d.get("label", "").lower() == label.lower()]
    d = (exact or docs)[0]
    desc = d.get("description") or [""]
    return {"curie": d.get("obo_id") or d.get("short_form"), "iri": d.get("iri"), "label": d.get("label"),
            "definition": (desc[0] if isinstance(desc, list) else desc)[:220],
            "source": d.get("ontology_name", "").upper(), "match": "exact" if exact else "review"}

def resolve_concepts():
    out = {}
    for key, (label, onts, cached) in CONCEPTS_WANTED.items():
        hit = ols_lookup(label, onts)
        if not hit:
            curie = cached or f"demo:{key}"
            hit = {"curie": curie, "iri": purl(curie), "label": label, "definition": "",
                   "source": onts.split(",")[0].upper(), "match": "cached" if cached else "unresolved"}
        out[key] = hit
    return out

# ---------------- Variables and context ----------------
# regex on column name -> (concept, mapping type, property, measured on, matrix)
RULES = [
    (r"height",                         "plant_height",       "skos:exactMatch",   "height",              "tomato plant",  None),
    (r"\blai\b|leaf.?area.?index",      "leaf_area_index",    "skos:exactMatch",   "leaf area index",     "tomato canopy", None),
    (r"leaf.?area",                     "leaf_area",          "skos:exactMatch",   "area",                "tomato leaf",   None),
    (r"dry.?matter|\bdmc\b",            "dry_matter_content", "skos:closeMatch",   "dry matter fraction", "tomato fruit",  None),
    (r"dry.?(mass|weight)|\bdw\b",      "plant_dry_weight",   "skos:closeMatch",   "dry mass",            "tomato plant organ", None),
    (r"yield|fresh.?(mass|weight)|\bfw\b|production", "fruit_yield", "skos:closeMatch", "fresh mass", "tomato fruit", None),
    (r"\btemp|temperature|\btair\b",    "temperature",        "skos:broadMatch",   "temperature",         "air",           "growing space"),
    (r"\brh\b|humid",                   "relative_humidity",  "skos:closeMatch",   "relative humidity",   "air",           "growing space"),
    (r"co2|co₂",                        "carbon_dioxide",     "skos:relatedMatch", "concentration",       "carbon dioxide","air in growing space"),
    (r"\bpar\b|ppfd|\bdli\b|light|radiation", "par",          "skos:closeMatch",   "light intensity / sum", "PAR",         "growing space"),
]
UNIT = re.compile(r"[\(\[]\s*([^\)\]]+?)\s*[\)\]]")

DATASET_CONTEXT = {   # None, or text with "confirm", counts as missing context
    PRIMARY_DOI: {"growing_system": "vertical farm, climate-controlled chambers",
                  "lighting": "artificial light only (expected in a vertical farm; confirm)",
                  "location": "Bleiswijk, NL", "period": "2021-2022, two trials",
                  "cultivars": "F12414 (red, plum); F13469B (orange, cherry)",
                  "design": "cultivation system x daylight integral x temperature x year"},
    COMPARISON_DOI: {"growing_system": "greenhouse compartment, 96 m2",
                     "lighting": "daylight, possibly supplemented (confirm)",
                     "location": "Bleiswijk, NL", "period": "two-month cultivation (dates: confirm)",
                     "cultivars": None, "design": None},
}
FILE_METHOD = {"CropParameters.csv": "destructive measurement"}
DOCUMENTED_VARIABLES = {   # only used when no file headers could be read
    PRIMARY_DOI: [("Production.csv", "fruit yield"), ("Production.csv", "fruit dry matter content"),
                  ("IndoorClimate.csv", "air temperature setting"), ("IndoorClimate.csv", "daylight integral"),
                  ("CropParameters.csv", "crop parameters (destructive)")],
    COMPARISON_DOI: [],
}

def describe_variables(doi, columns):
    pairs = [(f.split(" [")[0], c) for f, cols in columns.items() for c in cols]
    source = "file header"
    if not pairs:
        pairs, source = DOCUMENTED_VARIABLES[doi], "dataset description"
    out = []
    for fname, col in pairs:
        rule = next((r for r in RULES if re.search(r[0], col, flags=re.I)), None)
        unit = UNIT.search(col)
        rec = {"doi": doi, "dataset": LABEL[doi], "file": fname, "variable": col,
               "variable_id": f"{SHORT[doi]}:{fname}#{re.sub(r'[^A-Za-z0-9]+', '_', col).strip('_')}",
               "unit": unit.group(1) if unit else None, "method": FILE_METHOD.get(fname), "source": source,
               "concept": None, "predicate": None, "property": None, "measured_on": None, "matrix": None}
        if rule:
            rec.update(dict(zip(["concept", "predicate", "property", "measured_on", "matrix"], rule[1:])))
        out.append(rec)
    return out

# ---------------- Mappings (SSSOM) ----------------
PLAIN = {"skos:exactMatch": "same as", "skos:closeMatch": "close to", "skos:broadMatch": "is a kind of",
         "skos:narrowMatch": "is broader than", "skos:relatedMatch": "related to"}
AUTHOR = "orcid:0000-0000-0000-0000"   # replace with the curator's ORCID
MAPPING_SET = {
    "mapping_set_id": "https://example.org/digilab/demo/tomato-mappings.sssom.tsv",
    "mapping_set_version": "0.2",
    "license": "https://creativecommons.org/publicdomain/zero/1.0/",
    "curie_map": {"skos": "http://www.w3.org/2004/02/skos/core#", "semapv": "https://w3id.org/semapv/vocab/",
                  "orcid": "https://orcid.org/", "TO": "http://purl.obolibrary.org/obo/TO_",
                  "PATO": "http://purl.obolibrary.org/obo/PATO_", "ENVO": "http://purl.obolibrary.org/obo/ENVO_",
                  "CHEBI": "http://purl.obolibrary.org/obo/CHEBI_", "NCBITaxon": "http://purl.obolibrary.org/obo/NCBITaxon_",
                  "vf": f"https://doi.org/{PRIMARY_DOI}#", "gh": f"https://doi.org/{COMPARISON_DOI}#",
                  "demo": "https://example.org/digilab/demo/"},
}

def build_sssom(mapped, concepts):
    rows = []
    for _, v in mapped.iterrows():
        c = concepts[v["concept"]]
        rows.append({"subject_id": v["variable_id"], "subject_label": v["variable"], "predicate_id": v["predicate"],
                     "object_id": c["curie"], "object_label": c["label"],
                     "mapping_justification": "semapv:LexicalMatching", "author_id": AUTHOR,
                     "confidence": 0.6 if c["match"] in ("review", "unresolved") else 0.8,
                     "comment": "suggested from the column name; to be reviewed"})
    curated = [("vf:cultivar_F12414", "cultivar F12414", "skos:broadMatch", "tomato", "a cultivar is a kind of the species"),
               ("vf:cultivar_F13469B", "cultivar F13469B", "skos:broadMatch", "tomato", "a cultivar is a kind of the species"),
               (concepts["carbon_dioxide"]["curie"], concepts["carbon_dioxide"]["label"], "skos:relatedMatch", "ghg_emission",
                "same gas; a concentration indoors is not an emission")]
    for subj, slabel, pred, okey, note in curated:
        o = concepts[okey]
        rows.append({"subject_id": subj, "subject_label": slabel, "predicate_id": pred, "object_id": o["curie"],
                     "object_label": o["label"], "mapping_justification": "semapv:ManualMappingCuration",
                     "author_id": AUTHOR, "confidence": 0.95, "comment": note})
    return pd.DataFrame(rows)

def write_sssom(df, path):
    import yaml
    header = "".join(f"# {l}\n" for l in yaml.safe_dump(MAPPING_SET, sort_keys=False).splitlines())
    path.write_text(header + df.to_csv(sep="\t", index=False))
    return path

# ---------------- Comparison, driven by the mappings ----------------
def missing(value):
    return value is None or "confirm" in str(value)

def judge_mapping(preds):
    if "skos:relatedMatch" in preds:
        return "related, not comparable", "the link only says 'related to'"
    if "skos:broadMatch" in preds:
        return "check first", "mapped to a broader term: check what exactly was measured"
    if "skos:closeMatch" in preds:
        return "comparable with caveats", "'close to' on at least one side: compare the definitions"
    return "comparable with caveats", "'same as' on both sides"

def compare_concept(key, group, concepts):
    sides = []
    for doi, g in group.groupby("doi", sort=False):
        sides.append({"label": LABEL[doi], "vars": list(g["variable"]), "preds": set(g["predicate"]),
                      "units": set(g["unit"].dropna()), "methods": set(g["method"].dropna()),
                      "ctx": DATASET_CONTEXT[doi]})
    a, b = sides[0], sides[1]
    verdict, why = judge_mapping(a["preds"] | b["preds"])
    notes = [why]
    if not a["units"] or not b["units"]:
        notes.append("unit missing on one side")
        if verdict.startswith("comparable"):
            verdict = "check first"
    elif a["units"] != b["units"]:
        notes.append(f"different units ({', '.join(a['units'])} vs {', '.join(b['units'])}): same quantity?")
    for f in ["growing_system", "lighting"]:
        if not missing(a["ctx"][f]) and not missing(b["ctx"][f]) and a["ctx"][f] != b["ctx"][f]:
            notes.append(f"{f.replace('_', ' ')} differs")
    if a["methods"] != b["methods"]:
        notes.append("measurement method differs or is unknown")
    ask = [f.replace("_", " ") for f in ["growing_system", "lighting", "period", "cultivars", "design"]
           if missing(a["ctx"][f]) or missing(b["ctx"][f])]
    if verdict == "comparable with caveats" and len(notes) == 1 and not ask and "'same as'" in why:
        verdict = "comparable"
    c = concepts[key]
    chain = (f"{a['label']}: '{a['vars'][0]}' <b>{PLAIN[next(iter(a['preds']))]}</b> "
             f"{link(c['label'], c['iri'])} <b>{PLAIN[next(iter(b['preds']))]}</b> '{b['vars'][0]}' :{b['label']}")
    return {"key": key, "Concept": link(c["label"], c["iri"]), "How the mappings connect them": chain,
            "Verdict": verdict, "Why": "; ".join(notes), "Ask the data owner about": ", ".join(ask) or "-"}

VERDICT_COLOUR = {"comparable": "#2e7d32", "comparable with caveats": "#f9a825",
                  "check first": "#ef6c00", "related, not comparable": "#9e9e9e"}

# ---------------- Pictures ----------------
EDGE_STYLE = {"skos:exactMatch": ("-", 2.2), "skos:closeMatch": ("-", 1.2),
              "skos:broadMatch": ("--", 1.2), "skos:relatedMatch": (":", 1.6)}

def draw_graph(mapped, concepts, verdicts, n_specimens, n_climate):
    wrap = lambda s, w=20: textwrap.fill(str(s), w)
    keys = list(dict.fromkeys(list(mapped["concept"]) + ["carbon_dioxide"]))
    where = {k: set(mapped.loc[mapped["concept"] == k, "doi"]) for k in keys}
    def band(k):
        w = where[k]
        if w == {COMPARISON_DOI}: return 0
        if len(w) > 1: return 1
        return 3 if k == "carbon_dioxide" else 2
    keys.sort(key=band)
    pos = {COMPARISON_DOI: (0, 0.8), PRIMARY_DOI: (0, 0.25)}
    for i, k in enumerate(keys):
        pos[k] = (1, 1 - (i + 0.5) / len(keys))
    pos["tomato"], pos["ghg_emission"] = (2, 0.5), (2, pos["carbon_dioxide"][1])
    pos["specimens"], pos["climate"] = (3, 0.5), (3, pos["carbon_dioxide"][1])
    fig, ax = plt.subplots(figsize=(13, max(5, 0.55 * len(keys) + 2)))
    def edge(u, v, pred, colour="#555"):
        ls, lw = EDGE_STYLE.get(pred, ("-", 1))
        (x1, y1), (x2, y2) = pos[u], pos[v]
        ax.annotate("", xy=(x2 - 0.09, y2), xytext=(x1 + 0.09, y1),
                    arrowprops=dict(arrowstyle="-", color=colour, lw=lw, linestyle=ls, shrinkA=0, shrinkB=0))
    seen = set()
    for _, v in mapped.iterrows():
        if (v["doi"], v["concept"]) not in seen:
            seen.add((v["doi"], v["concept"])); edge(v["doi"], v["concept"], v["predicate"])
    edge(PRIMARY_DOI, "tomato", "skos:broadMatch")
    edge("carbon_dioxide", "ghg_emission", "skos:relatedMatch")
    edge("tomato", "specimens", "skos:exactMatch", "#1565c0")
    edge("ghg_emission", "climate", "skos:exactMatch", "#1565c0")
    def node(k, text, fc, ec="#333", bold=False):
        x, y = pos[k]
        ax.text(x, y, wrap(text), ha="center", va="center", fontsize=9, fontweight="bold" if bold else "normal",
                bbox=dict(boxstyle="round,pad=0.4", fc=fc, ec=ec, lw=1))
    for doi in SELECTED:
        node(doi, LABEL[doi], "#e3f2fd", bold=True)
    for k in keys:
        node(k, concepts[k]["label"], VERDICT_COLOUR.get(verdicts.get(k), "#f5f5f5") + ("" if k in verdicts else ""),
             ec="#333")
    node("tomato", "tomato (species)", "#f5f5f5"); node("ghg_emission", concepts["ghg_emission"]["label"], "#f5f5f5")
    node("specimens", f"{n_specimens} living tomato specimens in NL (GBIF)", "#e8f5e9", bold=True)
    node("climate", f"{n_climate} climate / emission datasets", "#e8f5e9", bold=True)
    for x, t in [(0, "Tomato datasets"), (1, "Terms in the data"), (2, "Linked terms"), (3, "Related data")]:
        ax.text(x, 1.06, t, ha="center", fontsize=11, fontweight="bold", color="#333")
    legend = [Line2D([], [], color="#555", lw=2.2, ls="-", label="same as"),
              Line2D([], [], color="#555", lw=1.2, ls="-", label="close to"),
              Line2D([], [], color="#555", lw=1.2, ls="--", label="is a kind of"),
              Line2D([], [], color="#555", lw=1.6, ls=":", label="related to"),
              Line2D([], [], color="#1565c0", lw=2, label="found via the term")]
    legend += [Line2D([], [], marker="s", ls="", ms=11, mfc=c, mec="#333", label=v)
               for v, c in VERDICT_COLOUR.items()]
    ax.legend(handles=legend, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=4, frameon=False, fontsize=9)
    ax.set_xlim(-0.35, 3.35); ax.set_ylim(-0.05, 1.12); ax.axis("off")
    ax.set_title("Two tomato datasets share terms; other domains connect through linked terms", fontsize=13, loc="left")
    plt.tight_layout(); plt.show()

def draw_architecture():
    fig, ax = plt.subplots(figsize=(12, 6.2))
    def box(x, y, w, h, title, sub, fc, steps=""):
        ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.02,rounding_size=0.08", fc=fc, ec="#444", lw=1.2))
        ax.text(x + w / 2, y + h * 0.64, title, ha="center", va="center", fontsize=11, fontweight="bold")
        ax.text(x + w / 2, y + h * 0.3, sub, ha="center", va="center", fontsize=9, color="#444")
        if steps:
            ax.text(x + w - 0.08, y + h - 0.1, steps, ha="right", va="top", fontsize=8.5, color="#1565c0", fontweight="bold")
    def arrow(x1, y1, x2, y2, both=False):
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="<|-|>" if both else "-|>", color="#666", lw=1.3))
    box(3.5, 5.0, 5, 0.9, "Researcher or AI agent", "asks one question, gets linked answers", "#e3f2fd", "")
    box(0.5, 3.0, 5.2, 1.3, "Federated metadata catalogue", "finds datasets, files and variables (DCAT/CDIF, RO-Crate)", "#fff8e1", "steps 1, 2, 6")
    box(6.3, 3.0, 5.2, 1.3, "Vocabulary hub", "explains terms and records links between them (SSSOM)", "#fff8e1", "steps 3, 4, 5, 7")
    box(0.5, 1.75, 11, 0.6, "Harvesting: descriptions are collected; data stays at the source", "", "#f5f5f5")
    box(0.5, 0.2, 3.4, 1.0, "Agriculture", "4TU.ResearchData (WUR)", "#e8f5e9")
    box(4.3, 0.2, 3.4, 1.0, "Environment", "climate and emission data", "#e8f5e9")
    box(8.1, 0.2, 3.4, 1.0, "Biodiversity", "GBIF, Naturalis", "#e8f5e9")
    arrow(5.0, 5.0, 3.1, 4.3); arrow(7.0, 5.0, 8.9, 4.3); arrow(5.7, 3.65, 6.3, 3.65, both=True)
    for x in (2.2, 6.0, 9.8):
        arrow(x, 1.2, x, 1.75)
    arrow(3.1, 2.35, 3.1, 3.0); arrow(8.9, 2.35, 8.9, 3.0)
    ax.plot([8.5, 12.1, 12.1], [5.45, 5.45, 0.7], color="#1565c0", lw=1.2, ls="--")
    ax.annotate("", xy=(11.5, 0.7), xytext=(12.1, 0.7),
                arrowprops=dict(arrowstyle="-|>", color="#1565c0", lw=1.2, linestyle="--"))
    ax.text(12.2, 3.1, "data fetched\nfrom the source", fontsize=8.5, color="#1565c0", va="center")
    ax.set_xlim(0, 13.4); ax.set_ylim(0, 6.1); ax.axis("off")
    ax.set_title("Two shared services connect independent repositories", fontsize=13, loc="left")
    plt.tight_layout(); plt.show()

print("Live APIs reachable:", ONLINE, "" if ONLINE else "(using cached descriptions; results will be thinner)")

## 1. Search the catalogue

The researcher types a few keywords. The catalogue returns datasets from different organisations, each with a title, organisation, year and licence (can I reuse it?). The two dwarf tomato datasets (a vertical farm trial and a greenhouse trial) are taken forward.

In [ ]:
hits = {}
for q in SEARCH_QUERIES:
    for r in datacite_search(q, prefix=REPOSITORY_PREFIX):
        s = summarise(r)
        hits.setdefault(s["doi"], s)
for doi in SELECTED:                           # make sure both demo datasets are there
    full = datacite_get(doi)
    hits[doi] = {**hits.get(doi, {}), **(summarise(full) if full else CACHED[doi])}

found = pd.DataFrame(hits.values())
found["selected"] = found["doi"].isin(SELECTED).map({True: "✔", False: ""})
found = found.sort_values(["selected", "year"], ascending=False)
found["Dataset"] = [link(t, u) for t, u in zip(found["title"], found["url"])]
print(f"{len(found)} datasets found")
found["Licence"] = [h if isinstance(h, str) and h else "not stated" for h in found.get("licence_html", pd.Series([""] * len(found)))]
show(found[["selected", "Dataset", "organisation", "year", "Licence"]].rename(columns={"selected": "", "organisation": "Organisation", "year": "Year"}))

<details><summary><b>Behind the scenes</b></summary>

**Target situation (once the DigiLab catalogue is in place):** each repository publishes a standard description of its datasets (DCAT/CDIF), and the federated catalogue harvests these descriptions, keeping the original identifier (DOI) so every hit points back to its source.

**In this demo we simulate that catalogue:** the DataCite search API plays its role, limited to 4TU.ResearchData. DataCite already holds a description of every DOI-registered dataset, so the search behaves much like the future catalogue would.
</details>

## 2. What's in this dataset?

The researcher opens a result and asks: *what's here, and is it worth downloading?* An AI agent asks the same question, but reads a machine-readable description instead of a web page. Both get the same answer: the files, their size, and what each file contains (its column names). Nothing is downloaded except the first lines of small files.

In [ ]:
CRATES, FILES, HEADS, COLUMNS = {}, {}, {}, {}
for doi in SELECTED:
    CRATES[doi] = Crate(fourtu_crate(doi))
    v2 = fourtu_files_v2(doi)
    files = CRATES[doi].files() or v2 or CACHED_FILES[doi]
    urls = {f.get("name"): f.get("download_url") for f in v2}
    for f in files:                       # fill download links missing from the crate
        f["download_url"] = f.get("download_url") or urls.get(f.get("name"))
    FILES[doi], HEADS[doi] = files, {}
    for f in files:
        HEADS[doi].update(read_head(f))
    COLUMNS[doi] = {k: [str(c) for c in d.columns] for k, d in HEADS[doi].items()}

for doi in SELECTED:
    display(Markdown(f"### {LABEL[doi]}"))
    show(dataset_card(CRATES[doi], hits[doi]), header=False)
    rows = [{"File": link(f.get("name"), f.get("download_url")), "Size": human_size(f.get("size")),
             "What's in it": describe_file(f, HEADS[doi])} for f in FILES[doi]]
    show(pd.DataFrame(rows)) if rows else print("  File list not available right now.")
    for label, d in HEADS[doi].items():
        details(f"Preview: first rows of {label}", d.head(3).to_html(index=False))

In [ ]:
# What an AI agent reads: the dataset's own RO-Crate (root entity, shortened)
crate = CRATES[PRIMARY_DOI]
if crate.root:
    agent_view = {k: v for k, v in crate.root.items() if k != "hasPart"}
    agent_view["hasPart"] = f"{len(as_list(crate.root.get('hasPart')))} parts"
else:
    agent_view = {"note": "RO-Crate not reachable; an agent would read the same facts shown above",
                  "files": [f.get("name") for f in FILES[PRIMARY_DOI]]}
details("What an AI agent reads for the vertical farm dataset (its RO-Crate, machine-readable)",
        f"<pre>{json.dumps(agent_view, indent=2, ensure_ascii=False)[:4000]}</pre>")

<details><summary><b>Behind the scenes</b></summary>

4TU.ResearchData already publishes an RO-Crate for every dataset version: files, sizes, creators, licence and more, in a standard machine-readable form. The dataset card and file list above are read from that RO-Crate (with the 4TU file API as fallback). For small CSV and Excel files (up to 15 MB) the first rows are previewed; large files and images are never downloaded.
</details>

## 3. What do the terms mean?

Column names in data files are short and often ambiguous: abbreviations, local naming, units squeezed into the name. The vocabulary hub links each column to a defined term, so the researcher can click through and see exactly what is meant, and an agent can follow the same link.

In [ ]:
CONCEPTS = resolve_concepts()
VARIABLES = pd.DataFrame([r for doi in SELECTED for r in describe_variables(doi, COLUMNS[doi])])
mapped = VARIABLES[VARIABLES["concept"].notna()].copy()

terms = mapped.drop_duplicates(["concept"]).copy()
terms["In the data"] = mapped.groupby("concept")["variable"].apply(lambda s: ", ".join(dict.fromkeys(s))).reindex(terms["concept"]).values
terms["Means"] = [link(CONCEPTS[k]["label"], CONCEPTS[k]["iri"]) for k in terms["concept"]]
terms["Definition"] = [CONCEPTS[k]["definition"] or "(follow the link)" for k in terms["concept"]]
show(terms[["In the data", "Means", "Definition"]])
print(f"{len(mapped)} of {len(VARIABLES)} columns linked to a defined term; the rest wait for a curator.")

In [ ]:
details("Behind the scenes: where the terms come from",
        pd.DataFrame([{"term": v["label"], "identifier": v["curie"], "vocabulary": v["source"], "match": v["match"]}
                      for v in CONCEPTS.values()]).to_html(index=False))

<details><summary><b>Behind the scenes</b></summary>

The **vocabulary hub** is an architecture component of its own: it holds the terms (from community vocabularies such as the Plant Trait Ontology, PATO, ENVO, ChEBI and the NCBI Taxonomy) and the links between them. Here the terms are looked up live in the EBI Ontology Lookup Service. Matches marked *review* need a person to confirm them.
</details>

## 4. What was measured, and under which conditions?

A number only makes sense with its context: what was measured, on what, in which unit, and where. This is what turns a column into something a researcher can interpret.

In [ ]:
view = mapped.copy()
view["Measured"] = [link(CONCEPTS[k]["label"], CONCEPTS[k]["iri"]) for k in view["concept"]]
view["Setting"] = [DATASET_CONTEXT[d]["growing_system"] for d in view["doi"]]
show(view[["dataset", "variable", "Measured", "measured_on", "unit", "Setting"]]
     .rename(columns={"dataset": "Dataset", "variable": "Column", "measured_on": "On", "unit": "Unit"}))

ctx = pd.DataFrame([{"Dataset": LABEL[d], **{k.replace("_", " ").capitalize(): (v or "unknown") for k, v in DATASET_CONTEXT[d].items()}}
                    for d in SELECTED])
display(Markdown("**Experimental context** (from the dataset descriptions; *confirm* and *unknown* = ask the data owner)"))
show(ctx)

In [ ]:
details("Behind the scenes: full variable description (for data modellers)",
        VARIABLES[["dataset", "file", "variable", "property", "measured_on", "matrix", "unit", "method", "source"]]
        .rename(columns={"measured_on": "object of interest"}).to_html(index=False, na_rep=""))

<details><summary><b>Behind the scenes</b></summary>

Each variable is broken down into *property* (what), *object of interest* (on what), *matrix* (in what) and *context*, following the RDA I-ADOPT framework. Column-to-term matching uses simple, visible rules on the column names; in DigiLab this would be curated in the vocabulary hub.
</details>

## 5. How are terms linked, and who says so?

Saying "these two things are the same" is a scientific claim. **SSSOM** (Simple Standard for Sharing Ontological Mappings) records every such link as one row that says:

- **what** is linked to what;
- **how strongly**: *same as*, *close to*, *is a kind of*, or only *related to*;
- **who** asserted it, **how** (automatic suggestion or expert review) and with what **confidence**.

Because each link is explicit, versioned and reviewable, the catalogue can explain *why* it shows two datasets together, and a curator can correct a wrong link without touching the data. Step 7 uses exactly these rows to decide what is comparable.

In [ ]:
SSSOM = build_sssom(mapped, CONCEPTS)
sssom_path = write_sssom(SSSOM, OUT / "tomato-mappings.sssom.tsv")

readable = pd.DataFrame({
    "From": SSSOM["subject_label"],
    "Link": SSSOM["predicate_id"].map(PLAIN),
    "To": [link(l, purl(c)) for l, c in zip(SSSOM["object_label"], SSSOM["object_id"])],
    "Asserted": SSSOM["mapping_justification"].map({"semapv:LexicalMatching": "suggested automatically",
                                                    "semapv:ManualMappingCuration": "expert-curated"}),
    "Confidence": SSSOM["confidence"],
    "Note": SSSOM["comment"],
})
show(readable)
print(f"Saved as an SSSOM file: {sssom_path}")

In [ ]:
details("Behind the scenes: the SSSOM file itself (first lines)",
        "<pre>" + "\n".join(sssom_path.read_text().splitlines()[:24]) + "</pre>")

## 6. Find related data in other domains

The links in the vocabulary hub lead beyond agriculture:

- the cultivars **are a kind of** the species *Solanum lycopersicum*, which finds **living tomato specimens** held in the Netherlands (for example in botanical gardens and gene banks) through GBIF;
- indoor CO₂ is **related to** greenhouse-gas emissions, which finds **climate and emission datasets** in the catalogue.

In [ ]:
GBIF = "https://api.gbif.org/v1/occurrence/search"
g = get_json(GBIF, {"scientificName": "Solanum lycopersicum", "basisOfRecord": "LIVING_SPECIMEN",
                    "country": "NL", "limit": 10}) if ONLINE else None
specimens = pd.DataFrame([{
    "Specimen": link(r.get("catalogNumber") or r.get("key"), f"https://www.gbif.org/occurrence/{r.get('key')}"),
    "Name": r.get("scientificName"), "Held by": r.get("institutionCode") or r.get("datasetName"),
    "Place": r.get("locality") or r.get("stateProvince"), "Year": r.get("year"),
    "Licence": link(licence_label(r.get("license")), r.get("license"))}
    for r in (g or {}).get("results", [])])
n_specimens = (g or {}).get("count", 0)
display(Markdown(f"**Living tomato specimens in the Netherlands (GBIF):** {n_specimens} records; first {len(specimens)} shown"))
show(specimens) if not specimens.empty else print("  none retrieved")

climate = {}
for q in ['"greenhouse gas" AND emission AND Netherlands', '"carbon dioxide" AND greenhouse AND climate']:
    for r in datacite_search(q, size=5):
        s = summarise(r); climate.setdefault(s["doi"], s)
climate_df = pd.DataFrame(climate.values())
display(Markdown(f"**Climate and emission datasets in the catalogue:** {len(climate_df)}"))
if not climate_df.empty:
    climate_df["Dataset"] = [link(t, u) for t, u in zip(climate_df["title"], climate_df["url"])]
    climate_df["Licence"] = [h or "not stated" for h in climate_df["licence_html"]]
    show(climate_df[["Dataset", "organisation", "year", "Licence"]].rename(columns={"organisation": "Organisation", "year": "Year"}))
else:
    print("  none retrieved")

## 7. What can be compared, and what context is missing?

Related is not the same as comparable. For every term that appears in both tomato datasets, the verdict follows from the SSSOM links (step 5) and the context (step 4):

| SSSOM link on both sides | Starting verdict |
|---|---|
| *same as* | comparable, if units and context agree |
| *close to* | comparable with caveats: compare definitions |
| *is a kind of* | check first: the term is broader than what was measured |
| *related to* | related, not comparable |

The last column is the extra context the researcher should ask the data owner for.

In [ ]:
shared = mapped.groupby("concept").filter(lambda g: g["doi"].nunique() > 1)
comparisons = [compare_concept(k, g, CONCEPTS) for k, g in shared.groupby("concept")]
comparisons += [
    {"key": "carbon_dioxide", "Concept": link(CONCEPTS["carbon_dioxide"]["label"], CONCEPTS["carbon_dioxide"]["iri"]),
     "How the mappings connect them": f"indoor CO₂ <b>related to</b> {CONCEPTS['ghg_emission']['label']}",
     "Verdict": "related, not comparable",
     "Why": "same gas, different quantity: indoor concentration vs emission to the atmosphere",
     "Ask the data owner about": "quantity, unit, spatial scale, system boundary"},
    {"key": "tomato", "Concept": link("Solanum lycopersicum", CONCEPTS["tomato"]["iri"]),
     "How the mappings connect them": "trial cultivars <b>is a kind of</b> tomato (species) → living specimens",
     "Verdict": "related, not comparable",
     "Why": "bred dwarf cultivars in controlled growth vs living collection plants",
     "Ask the data owner about": "cultivar, growing conditions, whether traits were measured"},
]
COMPARISON = pd.DataFrame(comparisons)
VERDICTS = {r["key"]: r["Verdict"] for r in comparisons}
if shared.empty:
    print("No term found in both tomato datasets yet (offline, or file headers not readable).")

styled = COMPARISON.drop(columns="key").copy()
styled["Verdict"] = [f'<span style="background:{VERDICT_COLOUR[v]};color:white;padding:2px 6px;border-radius:4px">{v}</span>'
                     for v in styled["Verdict"]]
show(styled)

display(Markdown("**Reuse conditions.** Comparable data is only useful if you may combine it:"))
reuse = []
for doi in SELECTED:
    name, url = CRATES[doi].licence()
    reuse.append({"Dataset": LABEL[doi],
                  "Licence": link(name, url) if name else (hits[doi].get("licence_html") or "not stated: ask the data owner")})
show(pd.DataFrame(reuse))

## Summary

**The picture:** both tomato datasets connect through shared terms. Colour shows what can be compared, line style shows the type of link, and the blue arrows show related data found through those links.

In [ ]:
draw_graph(mapped, CONCEPTS, VERDICTS, n_specimens, len(climate_df))

In [ ]:
lines = [f"**Found** {len(found)} datasets; two dwarf tomato datasets selected: "
         f"{LABEL[PRIMARY_DOI].lower()} and {LABEL[COMPARISON_DOI].lower()} (both WUR).",
         f"**Understood** {len(mapped)} of {len(VARIABLES)} columns through defined terms; the rest wait for a curator.",
         f"**Related data**: {n_specimens} living tomato specimens in NL (GBIF), {len(climate_df)} climate or emission datasets.",
         "", "**What can be compared:**"]
for r in comparisons:
    label = re.sub("<[^>]+>", "", r["Concept"])
    lines.append(f"- *{label}*: **{r['Verdict']}**. {r['Why'].capitalize()}."
                 + (f" Ask about: {r['Ask the data owner about']}." if r["Ask the data owner about"] != "-" else ""))
lines += ["", "Nothing was merged. Every result points back to its source, and every link is a reviewable SSSOM row."]
display(Markdown("\n".join(lines)))

**How it fits together:** the researcher or agent talks to two shared services. The catalogue finds datasets; the vocabulary hub explains and links terms. Repositories keep their data and only share descriptions. Blue labels show which demo steps use which service.

In [ ]:
draw_architecture()